# 9주차 — 의사결정나무 (Decision Tree) 실습  ·  2026 개정판

기계학습특론 | 성명: 최승호 | 이메일: jcn99250@naver.com

## 실습 구성
1. **데이터 준비** — 서울시 구/동 위경도 → 4개 권역 분류 (파일이 없으면 자동 생성)
2. **scikit-learn 의사결정나무** — `plot_tree`(graphviz 불필요), 결정 경계, 규칙 텍스트
3. **엔트로피·정보이득 직접 구현** — 컴퓨터 구매 여부 14건으로 ID3 계산표 재현
4. **과적합과 가지치기** — 깊이-정확도 곡선, 비용-복잡도 가지치기(`ccp_alpha`) + 교차검증
5. **앙상블** — RandomForest · HistGradientBoosting · XGBoost 비교
6. **해석** — MDI vs 순열 중요도, SHAP(TreeSHAP)
7. **회귀나무** — 프로야구 연차·안타 → 연봉 예제

## 환경
- Python 3.11+, **scikit-learn ≥ 1.4**, (선택) xgboost ≥ 2.0, shap ≥ 0.45
- Colab: 5절의 xgboost, 6절의 shap 은 설치 셀을 실행 (없어도 나머지는 동작)

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, sklearn
print("python  :", sys.version.split()[0])
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)
print("sklearn :", sklearn.__version__)

try:
    import matplotlib.font_manager as fm
    for name in ["NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]:
        if any(name in f.name for f in fm.fontManager.ttflist):
            plt.rcParams["font.family"] = name; break
except Exception:
    pass
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 0
PALETTE = ["#13A4B4", "#E9A23B", "#D9534F", "#3C9D5D", "#7B5EA7"]

---
# 1. 데이터 준비 — 서울시 구/동 권역 분류

기존 실습과 동일하게 **경도·위도** 두 특징으로 **강북 / 강동 / 강서 / 강남** 4개 권역을 분류한다.
`district_dict_list.csv`(구 단위, 학습)와 `dong_dict_list.csv`(동 단위, 테스트)가 없으면
같은 성격의 데이터를 생성하므로 그대로 실행하면 된다.

In [ ]:
from pathlib import Path
from sklearn.datasets import make_blobs

FEATURES = ["longitude", "latitude"]
CENTERS = {"강북": (126.90, 37.60), "강동": (127.10, 37.57),
           "강서": (126.87, 37.50), "강남": (127.06, 37.50)}

def synth(n_per, std, seed):
    Xs, ys = [], []
    rng = np.random.default_rng(seed)
    for name, (lon, lat) in CENTERS.items():
        Xs.append(rng.normal([lon, lat], [std, std * 0.6], size=(n_per, 2)))
        ys += [name] * n_per
    df = pd.DataFrame(np.vstack(Xs), columns=FEATURES)
    df["label"] = ys
    return df

if Path("district_dict_list.csv").exists():
    train = pd.read_csv("district_dict_list.csv")
    test  = pd.read_csv("dong_dict_list.csv")
    print("파일에서 로드")
else:
    train = synth(n_per=15, std=0.060, seed=RANDOM_STATE)     # 구 단위(적은 표본)
    test  = synth(n_per=40, std=0.060, seed=RANDOM_STATE + 1) # 동 단위(많은 표본, 경계가 겹침)
    train.to_csv("district_dict_list.csv", index=False)
    test.to_csv("dong_dict_list.csv", index=False)
    print("파일이 없어 생성 후 저장")

X_train, y_train = train[FEATURES], train["label"]
X_test,  y_test  = test[FEATURES],  test["label"]
print("train:", X_train.shape, " test:", X_test.shape)
train.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for k, name in enumerate(CENTERS):
    m = y_train == name
    ax[0].scatter(X_train.longitude[m], X_train.latitude[m], c=PALETTE[k], s=40, edgecolor="k", lw=0.3, label=name)
    m = y_test == name
    ax[1].scatter(X_test.longitude[m], X_test.latitude[m], c=PALETTE[k], s=15, edgecolor="k", lw=0.2, label=name)
ax[0].set_title("학습 데이터 (구 단위)"); ax[1].set_title("테스트 데이터 (동 단위)")
for a in ax: a.set_xlabel("경도"); a.set_ylabel("위도"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

---
# 2. scikit-learn 의사결정나무

### 기존 실습과 달라진 점
| 기존 | 2026 개정 | 이유 |
|---|---|---|
| `LabelEncoder` 로 y 변환 후 `le.classes_[pred]` | **문자열 y 를 그대로 사용** | sklearn 트리는 문자열 라벨을 직접 지원 (`clf.classes_`) |
| `export_graphviz` + `graphviz` 패키지 | **`plot_tree`, `export_text`** | 추가 설치 불필요, Colab 에서 바로 렌더링 |
| 수동 `meshgrid` 결정 경계 | **`DecisionBoundaryDisplay`** (1.1+) | 코드 단순화 |
| `DecisionTreeClassifier()` 기본값 | `max_depth`, `min_samples_leaf` 명시 | 과적합 제어를 처음부터 의식 |

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf = DecisionTreeClassifier(criterion="gini", max_depth=3, min_samples_leaf=1,
                             random_state=RANDOM_STATE)
clf.fit(X_train, y_train)                      # y 가 문자열이어도 그대로 학습됨

pred = clf.predict(X_test)
print("클래스 :", list(clf.classes_))
print("훈련 정확도 :", round(clf.score(X_train, y_train), 3))
print("테스트 정확도 :", round(accuracy_score(y_test, pred), 3))
print()
print(classification_report(y_test, pred))
pd.DataFrame(confusion_matrix(y_test, pred, labels=clf.classes_),
             index=[f"실제 {c}" for c in clf.classes_],
             columns=[f"예측 {c}" for c in clf.classes_])

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
plot_tree(clf, feature_names=FEATURES, class_names=list(clf.classes_),
          filled=True, rounded=True, impurity=True, fontsize=9, ax=ax)
ax.set_title("의사결정나무 (max_depth=3)")
plt.show()

print(export_text(clf, feature_names=FEATURES))     # 규칙을 텍스트로

**plot_tree 읽는 법**

각 노드에 표시되는 항목:
- `longitude <= 126.988` — 분할 조건 (참이면 왼쪽, 거짓이면 오른쪽)
- `gini = 0.75` — 그 노드의 불순도 (0이면 순수)
- `samples = 32` — 노드에 도달한 관측치 수
- `value = [8, 8, 8, 8]` — 클래스별 개수 (`clf.classes_` 순서)
- `class = 강남` — 다수 클래스 (잎이면 예측값)
- 색: 다수 클래스, 진하기: 순도

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for a, depth in zip(ax, [3, None]):
    m = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    DecisionBoundaryDisplay.from_estimator(m, X_train, ax=a, alpha=0.25,
                                           response_method="predict", cmap="Set2")
    for k, name in enumerate(CENTERS):
        sel = y_train == name
        a.scatter(X_train.longitude[sel], X_train.latitude[sel], c=PALETTE[k], s=40, edgecolor="k", lw=0.3)
    a.set_title(f"max_depth={depth}  ·  test acc = {m.score(X_test, y_test):.3f}")
    a.set_xlabel("경도"); a.set_ylabel("위도")
plt.tight_layout(); plt.show()
print("→ 분할이 축에 평행하므로 결정 경계가 계단 모양이 된다 (축 정렬 편향).")

---
# 3. 엔트로피 · 정보이득 직접 구현 (ID3 계산표 재현)

강의 자료의 **컴퓨터 구매 여부 데이터(14건)** 로 손계산 결과를 코드로 재현한다.
- `H(D) = 0.940`
- `Gain(D, 나이) = 0.247` (최대) → 첫 분할 속성 = 나이
- 청소년 부분집합 → 학생 여부, 중년 부분집합 → 신용 등급

In [ ]:
URL = "https://raw.githubusercontent.com/AugustLONG/ML01/master/01decisiontree/AllElectronics.csv"
try:
    df = pd.read_csv(URL).drop("RID", axis=1)
    print("URL 에서 로드")
except Exception as e:
    print("URL 접근 실패 → 내장 데이터 사용:", type(e).__name__)
    rows = [("youth","high","no","fair","no"),   ("youth","high","no","excellent","no"),
            ("middle_aged","high","no","fair","yes"), ("senior","medium","no","fair","yes"),
            ("senior","low","yes","fair","yes"), ("senior","low","yes","excellent","no"),
            ("middle_aged","low","yes","excellent","yes"), ("youth","medium","no","fair","no"),
            ("youth","low","yes","fair","yes"), ("senior","medium","yes","fair","yes"),
            ("youth","medium","yes","excellent","yes"), ("middle_aged","medium","no","excellent","yes"),
            ("middle_aged","high","yes","fair","yes"), ("senior","medium","no","excellent","no")]
    df = pd.DataFrame(rows, columns=["age","income","student","credit_rating","class_buys_computer"])

TARGET = "class_buys_computer"
ATTRS = [c for c in df.columns if c != TARGET]
KOR = {"age": "나이", "income": "수입", "student": "학생", "credit_rating": "신용"}
print(df.shape)
df

In [ ]:
def entropy(y):
    """H(D) = -Σ p log2 p   (p = 0 인 항은 0·log0 = 0 으로 처리)"""
    p = y.value_counts(normalize=True).to_numpy()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def cond_entropy(data, attr):
    """H_A(D) = Σ_v (|D_v| / |D|) · H(D_v)"""
    n = len(data)
    return float(sum(len(sub) / n * entropy(sub[TARGET]) for _, sub in data.groupby(attr)))


def info_gain(data, attr):
    return entropy(data[TARGET]) - cond_entropy(data, attr)


def split_info(data, attr):
    """C4.5 의 SplitInfo — 속성값 분포 자체의 엔트로피 (값이 많은 속성에 벌점)"""
    p = data[attr].value_counts(normalize=True).to_numpy()
    return float(-(p * np.log2(p)).sum())


def gini(y):
    p = y.value_counts(normalize=True).to_numpy()
    return float(1 - (p ** 2).sum())


print("H(D) =", round(entropy(df[TARGET]), 3), "  (강의 자료: 0.940)")
print("Gini(D) =", round(gini(df[TARGET]), 3))

In [ ]:
def gain_table(data):
    """모든 후보 속성의 정보이득 · 이득비 · 지니 감소를 한 표로"""
    base_H, base_G, n = entropy(data[TARGET]), gini(data[TARGET]), len(data)
    rows = []
    for a in [c for c in data.columns if c != TARGET]:
        g = info_gain(data, a)
        si = split_info(data, a)
        dist = " · ".join(f"{v}({(sub[TARGET] == data[TARGET].unique()[0]).sum()},"
                          f"{len(sub) - (sub[TARGET] == data[TARGET].unique()[0]).sum()})"
                          for v, sub in data.groupby(a))
        rows.append({"속성": KOR.get(a, a), "값별 분포": dist,
                     "H_A(D)": round(cond_entropy(data, a), 3),
                     "Gain": round(g, 3),
                     "SplitInfo": round(si, 3),
                     "GainRatio": round(g / si, 3) if si > 0 else np.nan,
                     "ΔGini": round(base_G - sum(len(s) / n * gini(s[TARGET])
                                                 for _, s in data.groupby(a)), 3)})
    return pd.DataFrame(rows).sort_values("Gain", ascending=False).reset_index(drop=True)

print(f"루트 노드:  H(D) = {entropy(df[TARGET]):.3f},  n = {len(df)}")
gain_table(df)

In [ ]:
root = gain_table(df).iloc[0]["속성"]
root_attr = [a for a in ATTRS if KOR.get(a, a) == root][0]
print("첫 번째 분할 속성 =", root, f"({root_attr})")
print("=" * 78)

for v, sub in df.groupby(root_attr):                    # 자식 노드에서 다시 계산
    H = entropy(sub[TARGET])
    print(f"\n[{root_attr} = {v}]  n = {len(sub)},  H = {H:.3f},  분포 = {dict(sub[TARGET].value_counts())}")
    if H == 0:
        print("    → 순수 노드 (잎으로 종료)")
    else:
        sub2 = sub.drop(columns=[root_attr])
        t = gain_table(sub2)
        print(t[["속성", "Gain", "GainRatio"]].to_string(index=False))
        print(f"    → 두 번째 분할 속성 = {t.iloc[0]['속성']}")

### 3-1. 이론 검증 — 정보이득은 항상 ≥ 0 (= 상호정보량)

`Gain(D, A) = H(Y) − H(Y|A) = I(Y; A) ≥ 0` 이며,
sklearn 의 `mutual_info_score` 와 (자연로그 → log2 변환 후) 일치해야 한다.

In [ ]:
from sklearn.metrics import mutual_info_score

print(f"{'속성':<8}{'직접 계산 Gain':>16}{'mutual_info_score':>20}")
for a in ATTRS:
    g = info_gain(df, a)
    mi = mutual_info_score(df[a], df[TARGET]) / np.log(2)        # nats → bits
    print(f"{KOR[a]:<8}{g:>16.6f}{mi:>20.6f}")
    assert abs(g - mi) < 1e-9
    assert g >= -1e-12                                          # 항상 0 이상

# 무작위 데이터 100개에서도 Gain >= 0 인지 확인
rng = np.random.default_rng(RANDOM_STATE)
mins = []
for _ in range(100):
    tmp = pd.DataFrame({"a": rng.integers(0, 4, 200), TARGET: rng.integers(0, 3, 200)})
    mins.append(info_gain(tmp, "a"))
print("\n무작위 데이터 100회의 최소 Gain :", round(min(mins), 6), "  → 항상 0 이상 ✔")

### 3-2. sklearn 으로 같은 트리 만들기

CART 는 **이진 분할**이므로 다지 분할하는 ID3 와 트리 모양이 다를 수 있지만,
`criterion='entropy'` 로 두면 같은 기준(정보이득)을 사용한다.

In [ ]:
X_cat = pd.get_dummies(df[ATTRS])                  # 범주형 → 원-핫
y_cat = df[TARGET]

tree_e = DecisionTreeClassifier(criterion="entropy", random_state=RANDOM_STATE).fit(X_cat, y_cat)
tree_g = DecisionTreeClassifier(criterion="gini",    random_state=RANDOM_STATE).fit(X_cat, y_cat)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, (m, name) in zip(ax, [(tree_e, "entropy"), (tree_g, "gini")]):
    plot_tree(m, feature_names=list(X_cat.columns), class_names=list(m.classes_),
              filled=True, rounded=True, fontsize=7, ax=a)
    a.set_title(f"criterion='{name}'  ·  잎 {m.get_n_leaves()}개  ·  훈련 정확도 {m.score(X_cat, y_cat):.2f}")
plt.tight_layout(); plt.show()

print("두 기준이 같은 예측을 하는가:", bool((tree_e.predict(X_cat) == tree_g.predict(X_cat)).all()))

---
# 4. 과적합과 가지치기

1. 깊이에 따른 훈련/테스트 정확도 곡선
2. 비용-복잡도 가지치기: `cost_complexity_pruning_path` 로 α 경로를 얻고 **교차검증**으로 선택

$$R_\alpha(T) = R(T) + \alpha \cdot |\text{leaves}(T)|$$

In [ ]:
depths = range(1, 16)
tr, te = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(X_train, y_train)
    tr.append(m.score(X_train, y_train)); te.append(m.score(X_test, y_test))

plt.figure(figsize=(7, 4))
plt.plot(list(depths), tr, "-o", color="#13A4B4", label="훈련 정확도")
plt.plot(list(depths), te, "-o", color="#D9534F", label="테스트 정확도")
plt.axvline(list(depths)[int(np.argmax(te))], color="#E9A23B", ls="--", label="최적 깊이")
plt.xlabel("max_depth"); plt.ylabel("정확도"); plt.title("트리 깊이와 과적합"); plt.legend(); plt.show()
print("최적 깊이 :", list(depths)[int(np.argmax(te))], " 테스트 정확도 :", round(max(te), 3))

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

path = DecisionTreeClassifier(random_state=RANDOM_STATE).cost_complexity_pruning_path(X_train, y_train)
alphas = np.unique(path.ccp_alphas)
alphas = alphas[alphas >= 0]

cv = StratifiedKFold(4, shuffle=True, random_state=RANDOM_STATE)
cv_scores, leaves, test_scores = [], [], []
for a in alphas:
    m = DecisionTreeClassifier(ccp_alpha=a, random_state=RANDOM_STATE)
    cv_scores.append(cross_val_score(m, X_train, y_train, cv=cv).mean())
    m.fit(X_train, y_train)
    leaves.append(m.get_n_leaves()); test_scores.append(m.score(X_test, y_test))

best_alpha = alphas[int(np.argmax(cv_scores))]
pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=RANDOM_STATE).fit(X_train, y_train)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(alphas, leaves, "-o", color="#7B5EA7"); ax[0].set(xlabel="ccp_alpha", ylabel="잎 노드 수", title="α 증가 → 트리 단순화")
ax[1].plot(alphas, cv_scores, "-o", color="#13A4B4", label="교차검증(훈련)")
ax[1].plot(alphas, test_scores, "-o", color="#D9534F", label="테스트")
ax[1].axvline(best_alpha, color="#E9A23B", ls="--"); ax[1].set(xlabel="ccp_alpha", ylabel="정확도", title="비용-복잡도 가지치기"); ax[1].legend()
plt.tight_layout(); plt.show()

print(f"선택된 alpha = {best_alpha:.5f}  |  잎 {pruned.get_n_leaves()}개  |  테스트 정확도 {pruned.score(X_test, y_test):.3f}")

---
# 5. 앙상블 — RandomForest · HistGradientBoosting · XGBoost

- **배깅/RF**: 완전히 자란 트리의 **분산**을 평균으로 줄임 → `Var = ρσ² + (1−ρ)σ²/B`
- **부스팅**: 얕은 트리를 순차적으로 더해 **편향**을 줄임 (조기 종료 필수)

In [ ]:
# Colab: XGBoost 가 없으면 아래 주석 해제
# !pip install -q xgboost
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

y_tr_i, classes = pd.factorize(y_train)
y_te_i = pd.Categorical(y_test, categories=classes).codes

models = {
    "tree(pruned)": pruned,
    "random_forest": RandomForestClassifier(500, max_features="sqrt", oob_score=True,
                                            n_jobs=-1, random_state=RANDOM_STATE),
    "hist_gb": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300,
                                              early_stopping=True, validation_fraction=0.25,
                                              random_state=RANDOM_STATE),
}
try:
    from xgboost import XGBClassifier
    models["xgboost"] = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                                      subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                                      random_state=RANDOM_STATE)
except ImportError:
    print("xgboost 가 설치되어 있지 않습니다 — 해당 모델은 건너뜁니다.")

rows = []
for name, m in models.items():
    if name == "tree(pruned)":
        acc = m.score(X_test, y_test)
    else:
        m.fit(X_train, y_tr_i)
        acc = float((m.predict(X_test) == y_te_i).mean())
    rows.append({"model": name, "test_accuracy": round(acc, 3)})
res = pd.DataFrame(rows).set_index("model")
print("RF OOB score :", round(models["random_forest"].oob_score_, 3))
res

In [ ]:
# 앙상블의 결정 경계는 단일 트리보다 매끄럽다
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
show = [("단일 트리(가지치기)", pruned, y_train),
        ("랜덤 포레스트", models["random_forest"], y_tr_i),
        ("HistGradientBoosting", models["hist_gb"], y_tr_i)]
for a, (title, m, yy) in zip(ax, show):
    DecisionBoundaryDisplay.from_estimator(m, X_train, ax=a, alpha=0.25,
                                           response_method="predict", cmap="Set2")
    for k, name in enumerate(CENTERS):
        sel = (y_train == name).to_numpy()
        a.scatter(X_train.longitude[sel], X_train.latitude[sel], c=PALETTE[k], s=35, edgecolor="k", lw=0.3)
    a.set_title(title); a.set_xlabel("경도")
ax[0].set_ylabel("위도")
plt.tight_layout(); plt.show()

---
# 6. 해석 — 특성 중요도와 SHAP

- **MDI** (`feature_importances_`): 학습 중 불순도 감소의 합 — 고유값이 많은 연속형 속성에 **편향**
- **순열 중요도**: 테스트 데이터에서 값을 섞었을 때의 성능 하락 — 모델 무관, 편향 적음

잡음 특징 두 개(`random_num`, `random_bin`)를 일부러 넣어 두 방법의 차이를 확인한다.

In [ ]:
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(RANDOM_STATE)
X_tr_aug = X_train.copy(); X_te_aug = X_test.copy()
X_tr_aug["random_num"] = rng.normal(size=len(X_tr_aug)); X_te_aug["random_num"] = rng.normal(size=len(X_te_aug))
X_tr_aug["random_bin"] = rng.integers(0, 2, len(X_tr_aug)); X_te_aug["random_bin"] = rng.integers(0, 2, len(X_te_aug))

rf = RandomForestClassifier(500, random_state=RANDOM_STATE, n_jobs=-1).fit(X_tr_aug, y_tr_i)
pi = permutation_importance(rf, X_te_aug, y_te_i, n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1)

imp = pd.DataFrame({"MDI": rf.feature_importances_,
                    "permutation": pi.importances_mean,
                    "perm_std": pi.importances_std}, index=X_tr_aug.columns)
display_cols = imp.sort_values("permutation", ascending=False).round(4)
print(display_cols)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
imp["MDI"].plot.barh(ax=ax[0], color="#13A4B4"); ax[0].set_title("MDI (불순도 감소) 중요도")
ax[1].barh(imp.index, imp["permutation"], xerr=imp["perm_std"], color="#E9A23B"); ax[1].set_title("순열 중요도 (테스트)")
plt.tight_layout(); plt.show()
print("→ 순수 잡음인 random_num 이 MDI 에서는 무시할 수 없는 값을 받는다 (고유값이 많은 연속형 편향).")

In [ ]:
# TreeSHAP: 트리 구조를 이용해 Shapley 값을 다항 시간에 정확히 계산
# Colab: !pip install -q shap
try:
    import shap
    model_for_shap = models.get("xgboost", models["random_forest"])
    if "xgboost" not in models:
        model_for_shap = RandomForestClassifier(200, max_depth=4, random_state=RANDOM_STATE).fit(X_train, y_tr_i)
    explainer = shap.TreeExplainer(model_for_shap)
    sv = explainer.shap_values(X_test)
    sv0 = sv[..., 0] if isinstance(sv, np.ndarray) and sv.ndim == 3 else (sv[0] if isinstance(sv, list) else sv)
    shap.summary_plot(sv0, X_test, feature_names=FEATURES, show=True)
    print("클래스", classes[0], "에 대한 SHAP 요약 (양수 = 그 클래스로 미는 기여)")
except ImportError:
    print("shap 이 설치되어 있지 않습니다. 위 pip 주석을 해제한 뒤 실행하세요.")

---
# 7. 회귀나무 — 프로야구 연차·안타 → 연봉

강의 자료의 Step 1~3 을 코드로 재현한다.
- Step 1. 데이터 분할 (분할 기준 확인)
- Step 2. 영역별 평균 계산 → 잎의 예측값
- Step 3. 회귀나무 완성 → 새 관측치 예측

In [ ]:
from sklearn.tree import DecisionTreeRegressor

years = np.array([2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15])
hits  = np.array([20, 26, 23, 35, 30, 52, 38, 40, 60, 50, 90, 80, 92])
salary = np.array([23, 25, 20, 26, 24, 42, 38, 41, 50, 48, 85, 82, 88])    # 단위 M
Xr = np.c_[years, hits]

rt = DecisionTreeRegressor(max_depth=2, min_samples_leaf=2, random_state=RANDOM_STATE).fit(Xr, salary)
print(export_text(rt, feature_names=["연차", "안타"]))
print("새 선수 (연차 5, 안타 35) 예측 연봉 :", rt.predict([[5, 35]])[0].round(1), "M")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
xx, yy = np.meshgrid(np.linspace(0, 16, 200), np.linspace(10, 100, 200))
Z = rt.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax[0].contourf(xx, yy, Z, alpha=0.35, cmap="YlOrBr")
ax[0].scatter(years, hits, c=salary, cmap="YlOrBr", s=150, edgecolor="k")
for x, y, s in zip(years, hits, salary):
    ax[0].annotate(str(s), (x, y), ha="center", va="center", fontsize=7)
ax[0].set(xlabel="연차 (x1)", ylabel="안타 수 (x2)", title="영역 분할 (값 = 연봉, 단위 M)")
plot_tree(rt, feature_names=["연차", "안타"], filled=True, rounded=True, fontsize=9, impurity=False, ax=ax[1])
ax[1].set_title("회귀나무 (depth=2) — 잎 = 영역 평균")
plt.tight_layout(); plt.show()

In [ ]:
# 회귀나무는 조각별 상수 → 외삽이 불가능하다는 점 확인
x_grid = np.linspace(0, 25, 300)
pred_grid = rt.predict(np.c_[x_grid, np.full_like(x_grid, 50)])

plt.figure(figsize=(7, 3.5))
plt.plot(x_grid, pred_grid, color="#D9534F", lw=2, label="회귀나무 예측 (안타 = 50 고정)")
plt.axvspan(years.min(), years.max(), color="#13A4B4", alpha=0.12, label="훈련 데이터 범위")
plt.xlabel("연차"); plt.ylabel("예측 연봉 (M)"); plt.title("조각별 상수 · 외삽 불가")
plt.legend(); plt.show()

---
# 8. 정리와 과제

## 정리
1. 트리는 **축 정렬 재귀 분할 + 영역별 상수**이며, `plot_tree`·`export_text` 로 규칙을 그대로 읽을 수 있다 (2절).
2. 분할 속성은 **정보이득 = 상호정보량 ≥ 0** 으로 고르며, 3-1절에서 `mutual_info_score` 와 일치함을 확인했다.
3. 탐욕 성장은 스스로 멈추지 않으므로 **깊이 제한·비용복잡도 가지치기(교차검증)** 가 필요하다 (4절).
4. 단일 트리의 큰 분산은 **랜덤 포레스트**가, 편향은 **부스팅**이 보완한다 (5절).
5. MDI 는 편향될 수 있으므로 **순열 중요도·SHAP** 을 함께 본다 (6절).

## 제출 과제
1. **(필수)** 3절의 `gain_table` 을 이용해 '수입' 을 첫 분할 속성으로 강제했을 때 만들어지는 트리를 손으로 그리고, ID3 트리와 잎 개수를 비교하시오.
2. **(필수)** 4절의 `ccp_alpha` 경로에서 잎이 4개가 되는 α 를 찾고, 그때의 트리를 `plot_tree` 로 그려 해석하시오.
3. **(심화)** `random_bin` 대신 '관측치 번호(0~N)' 열을 넣고 MDI·순열 중요도가 어떻게 달라지는지 확인하고, 이것이 왜 ID 같은 속성에 대한 정보이득 편향과 같은 현상인지 설명하시오.
4. **(심화)** 본인 연구 데이터에 `HistGradientBoosting` 또는 XGBoost 를 적용하고, 교차검증 → 조기 종료 → 캘리브레이션(`CalibratedClassifierCV`) → 운영 임계값 선택(`TunedThresholdClassifierCV`) 까지의 파이프라인을 구성해 AUROC·AUPRC 와 함께 보고하시오.